
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

# Init model
# model là positional đầu tiên (KHÔNG dùng model_name)
# GROQ_API_KEY được đọc tự động từ .env
model = init_chat_model("groq:qwen/qwen3-32b")

In [9]:
from pydantic import BaseModel,Field

# Init structured output for movie
# Tên Structure (Tên Class)
# Khai báo thông tin trả về như thuộc tính của structure
# Thông tin: tên thông tin, kiểu dữ liệu, mô tả(để LLM hiểu là phải trả về thông tin có đặc điểm như nào)

class Movie(BaseModel):
    title:str = Field(description="Tiêu đề/tên của bộ film")
    year:int = Field(description="Năm bộ phim được xuất bản")
    director:str = Field(description="Đạo diễn của bộ phim")
    rating:float = Field(description="Điểm đánh giá của bộ film")

In [ ]:
# Result

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5839C6660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5839C7380>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', '

In [13]:
# Compare LLM with & without structured output
model.invoke("Hãy cung cấp cho tôi thông tin về bộ film Mắt biếc")

AIMessage(content='<think>\nOkay, so I need to provide information about the movie "Mắt Biếc" to the user. Let me start by recalling what I know about it. First, I remember that it\'s a Vietnamese film, based on a novel. The director is Victor Vũ, right? He\'s known for other movies like "Hồn Papa Da Con Gái" and maybe "Tôi Thấy Hoa Vàng Trên Cỏ Xanh". \n\nThe book it\'s adapted from is "Mắt Biếc" by Nguyễn Nhật Ánh. I think he\'s a popular author in Vietnam, especially for young adults. The story is about a boy and a girl, their childhood friendship turning into love. The title "Mắt Biếc" translates to "Green Eyes" in English, but I\'m not sure if that\'s a direct translation. The main characters are Hà and Phương, maybe? Hà is the boy, and Phương is the girl with green eyes. The girl moves away, and they grow up with a connection.\n\nThe film was released in 2019. It was a big hit in Vietnam, I believe. The cast includes some popular actors. I think Hà is played by Lương Thùy Linh or

In [24]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Hãy cung cấp cho tôi thông tin về bộ film Mắt biếc")

Movie(title='Mắt Biếc', year=2020, director='Victor Vu', rating=7.5)

In [25]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. I need to use the Movie function they provided. Let me check the parameters required: title, year, director, and rating. I know the title is "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it 8.8/10. So I should structure the function call with those details. Make sure all required fields are included. No need for any optional parameters here. Alright, that should cover it.\n', 'tool_calls': [{'id': '2f47fhk0q', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 231, 'total_tokens': 415, 'completion_time': 0.283316543, 'completion_tokens_details': {'reasoning

### Nested Structure

In [52]:
from pydantic import BaseModel, Field

# Init Actor (name, role)
class Actor(BaseModel):
    name: str
    role: str

# Init MovieDetails (title, year, cast, genres, budget)
class MovieDetails(BaseModel):
    title: str
    year: int
    cast:list[Actor]
    genres: list[str]
    budget: float | None

In [53]:
movie_details_model = model.with_structured_output(MovieDetails)
response =  movie_details_model.invoke("Hãy cung cấp cho tôi thông tin về bộ film Mắt biếc năm 2019 do Victor Vũ đạo diễn, là một trong những tiểu thuyết nổi tiếng nhất của nhà văn Nguyễn Nhật Ánh, phát hành lần đầu năm 1990")

In [54]:
response

MovieDetails(title='Mắt Biếc', year=2019, cast=[Actor(name='Bùi Thùy Linh', role='Phương Oanh'), Actor(name='Kelvin Bryant', role='Tùng'), Actor(name='Hồng Đạo', role='Ông Hoa')], genres=['Romance', 'Drama'], budget=None)

In [55]:
cast_list=[actor.name for actor in response.cast]
cast_list

['Bùi Thùy Linh', 'Kelvin Bryant', 'Hồng Đạo']

In [ ]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [56]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [57]:
agent = create_agent(
    model="gpt-5-mini",
    response_format=ContactInfo
)

In [62]:
result = agent.invoke({          # ✅ dùng ()
    "messages": [                 # ✅ "messages"
        {"role": "user", "content": "Thông tin contact của tôi Khắc Vũ là khacvu0505@gmail.com, phone: 0399999999"}
    ]
})

result

{'messages': [HumanMessage(content='Thông tin contact của tôi Khắc Vũ là khacvu0505@gmail.com, phone: 0399999999', additional_kwargs={}, response_metadata={}, id='af07256a-9a78-449f-b3f5-996e5faef6f9'),
  AIMessage(content='{"name":"Khắc Vũ","email":"khacvu0505@gmail.com","phone":"0399999999"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 209, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E1AGaAWrvor4s3Kyip1evudbrSeeH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5b7f-549c-7140-b3ed-c6312e0a7411-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2